# Track B — Circuit Tracing on Gemma-2 2B
### Physical-reasoning circuits in a pretrained LLM

**Dissertation:** *Do Learned World Models Develop Interpretable Causal Structure?*  
**Author:** Nishchay Govindraj (250059083) — MSc AI, City, University of London

This is a single self-contained notebook: run top to bottom. It installs `circuit-tracer`, loads Gemma-2 2B with GemmaScope transcoders, reproduces the paper's known-good example as a sanity check, then traces a starter set of physical-reasoning prompts.

---
### Before you run
1. **Runtime → Change runtime type → GPU** (T4 is fine; L4/A100 lets you use the larger transcoders).
2. **Accept the Gemma license:** visit https://huggingface.co/google/gemma-2-2b while logged in and click *Agree*. Gemma is gated — without this, the model-load cell will 403.
3. Have a HuggingFace access token ready (https://huggingface.co/settings/tokens). The login cell will prompt for it.

### Verification note (read once)
The **install, login, model-load, and transcoder** cells use API calls verified against the current `circuit-tracer` demo notebooks. The **attribution / visualisation / intervention** cells mirror `demos/circuit_tracing_tutorial.ipynb`; if the library's API has moved since, cross-check against that tutorial rather than trusting these cells blindly. Each such cell is marked ⚠️ VERIFY.

## 1  Environment check

In [ ]:
# Confirm a GPU is attached and report its memory budget.
# This determines which transcoder set is safe to load (see cell 4).
import subprocess, sys

try:
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total",
                                   "--format=csv,noheader"]).decode().strip()
    print("GPU detected:", out)
except Exception as e:
    print("No GPU detected — go to Runtime > Change runtime type > GPU.")
    print("Error:", e)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory: {gib:.1f} GiB")

GPU detected: NVIDIA L4, 23034 MiB
CUDA available: True
GPU memory: 22.0 GiB


## 2  Install circuit-tracer
Clones the canonical repo and installs it. ~2-3 min on Colab.

In [ ]:
# Verified install flow: clone the repo, then `pip install .` in its directory.
import os, subprocess

if not os.path.isdir("circuit-tracer"):
    subprocess.run(["git", "clone",
                    "https://github.com/safety-research/circuit-tracer.git"], check=True)
else:
    print("circuit-tracer already cloned.")

%pip install -q ./circuit-tracer
print("\nInstall complete. If pip reports a dependency conflict, restart the")
print("runtime (Runtime > Restart) and re-run from cell 1 — Colab pins some")
print("packages that circuit-tracer upgrades.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.0/182.0 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.6/968.6 kB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

## 3  Authenticate with HuggingFace
Gemma is a **gated** model. You must have accepted the license (see 'Before you run') and supply a token.

In [ ]:
# Prompts for your HF token in a hidden field. The token is NOT written to disk
# or printed — never hardcode it into the notebook.
from huggingface_hub import login, get_token
login(new_session=True)   # paste the NEW token when prompted

# verify the new token is the one now active
print("Token loaded:", get_token()[:8] + "..." if get_token() else "NONE")

Token loaded: NONE


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from huggingface_hub import whoami, HfApi

# 1. Confirm a token is actually active in this session
try:
    me = whoami()
    print("Logged in as:", me["name"])
except Exception as e:
    print("NOT logged in — the login() cell didn't take. Re-run it and paste your token.")
    raise

# 2. Confirm THIS account can see the gated repo
api = HfApi()
try:
    info = api.model_info("google/gemma-2-2b")
    print("Access to google/gemma-2-2b: GRANTED")
except Exception as e:
    print("Access to google/gemma-2-2b: DENIED")
    print("-> You're authenticated, but this account hasn't been granted access.")
    print("-> Accept the license at https://huggingface.co/google/gemma-2-2b, then retry.")

Logged in as: Boozelightyear
Access to google/gemma-2-2b: GRANTED


## 4  Choose transcoder set (VRAM-aware)
The transcoders decompose MLP activations into interpretable features. Richer sets give finer circuits but cost more memory.

| Set | Features | Needs | Notes |
|---|---|---|---|
| `"gemma"` (GemmaScope PLTs) | per-layer | ~15 GB (T4 OK) | **default — safest** |
| 426K CLTs | 426K | ~15 GB (T4, tight) | cross-layer, better resolution |
| 2.5M CLTs | 2.5M | >15 GB (L4/A100) | richest; may OOM on T4 |

Default below is the safe path. To scale up on a bigger runtime, change `TRANSCODER_SET` to the CLT repo name and nothing else.

In [ ]:
# ── the ONE switch that trades resolution for memory ──
# Safe default (T4-friendly): the GemmaScope per-layer transcoders, aliased 'gemma'.
TRANSCODER_SET = "gemma"

# To use cross-layer transcoders on a larger runtime, comment the line above and
# uncomment ONE of these (confirm the exact repo name in the circuit-tracer README
# 'Available transcoders' table before relying on it):
# TRANSCODER_SET = "mntss/gemma-scope-transcoders"   # ⚠️ VERIFY exact repo/alias

MODEL_NAME = "google/gemma-2-2b"
print(f"Model:       {MODEL_NAME}")
print(f"Transcoders: {TRANSCODER_SET}")

Model:       google/gemma-2-2b
Transcoders: gemma


## 5  Load the ReplacementModel
Loads Gemma-2 2B wrapped so its MLPs are replaced by transcoder features. First load downloads weights (~5 GB) — a few minutes.

In [ ]:
# Verified call signature (from demos/gemma_demo.ipynb and the tutorial):
#   ReplacementModel.from_pretrained("google/gemma-2-2b", "gemma",
#                                    dtype=torch.bfloat16, backend=backend)
import os, torch
from huggingface_hub import get_token
from circuit_tracer import ReplacementModel

# Pass the token explicitly so the gated download can't miss it
tok = get_token()
os.environ["HF_TOKEN"] = tok            # some code paths read this env var
os.environ["HUGGING_FACE_HUB_TOKEN"] = tok  # older paths read this one

model = ReplacementModel.from_pretrained(
    MODEL_NAME,
    TRANSCODER_SET,
    dtype=torch.bfloat16,
)
print("Model + transcoders loaded.")

config.yaml:   0%|          | 0.00/202 [00:00<?, ?B/s]

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

layer_0.safetensors:   0%|          | 0.00/302M [00:00<?, ?B/s]

layer_1.safetensors:   0%|          | 0.00/302M [00:00<?, ?B/s]

layer_12.safetensors:   0%|          | 0.00/302M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
Model + transcoders loaded.


## 6  Sanity check — reproduce the paper's known-good circuit
Before tracing our own prompts, confirm the pipeline works on a result from the literature: *"Fact: The capital of the state containing Dallas is"* → the circuit should route Dallas → Texas → Austin. If this doesn't produce a sensible graph, stop and fix the setup before trusting any physics results.

⚠️ VERIFY: the exact `attribute` / graph-building call below mirrors `demos/circuit_tracing_tutorial.ipynb`. Cross-check argument names against that notebook on first run.

In [ ]:
# The next-token check is backend-agnostic and does NOT depend on the attribution
# API, so it is a safe first test that the model itself loaded correctly.
sanity_prompt = "Fact: The capital of the state containing Dallas is"

toks = model.tokenizer(sanity_prompt, return_tensors="pt").to(model.cfg.device \
        if hasattr(model, 'cfg') else 'cuda')
with torch.no_grad():
    logits = model(toks["input_ids"])
next_id = logits[0, -1].argmax().item()
print("Prompt:", sanity_prompt)
print("Predicted next token:", repr(model.tokenizer.decode([next_id])))
print("(Expect something like ' Austin'.)")

Prompt: Fact: The capital of the state containing Dallas is
Predicted next token: ' Austin'
(Expect something like ' Austin'.)


In [ ]:
# ⚠️ VERIFY against demos/circuit_tracing_tutorial.ipynb — the attribution API.
# The tutorial builds an attribution graph roughly as below. Argument names
# (max_n_logits, desired_logit_prob, batch_size, etc.) should be confirmed there;
# they are the parameters most likely to have changed across versions.
from circuit_tracer import attribute

def build_graph(prompt, **kw):
    """Wrapper so every trace in this notebook uses identical settings.
    Tune batch_size DOWN if you hit OOM on a T4."""
    return attribute(
        prompt=prompt,
        model=model,
        max_n_logits=10,
        desired_logit_prob=0.95,
        batch_size=1,          # conservative for T4; raise on bigger GPUs
        **kw,
    )

sanity_graph = build_graph(sanity_prompt)
print("Attribution graph built for the sanity prompt.")
print("Graph object:", type(sanity_graph))

sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


Attribution graph built for the sanity prompt.
Graph object: <class 'circuit_tracer.graph.Graph'>


## 7  Physical-reasoning prompt suite
The research target: does Gemma-2 2B contain identifiable circuits for simple physical reasoning, and do they show the distributed-but-causal character found in Track A?

Edit `PHYSICS_PROMPTS` freely. Each is written so the expected next token is a specific physical outcome, giving attribution a clear target. Grouped by the reasoning type they probe.

In [ ]:
# Starter suite — edit / extend. Keep the expected answer as (ideally) a single
# token so the attribution target is unambiguous.
PHYSICS_PROMPTS = {
    # Support / gravity
    "gravity_fall":  "If you let go of a ball in mid-air, it will move",          # down
    "support_remove":"A cup rests on a table. If the table is removed, the cup will", # fall
    # Collision / momentum
    "collision":     "A fast ball hits a still ball head-on. The still ball starts to", # move
    # Containment / spilling
    "tip_over":      "A full glass of water is tipped sideways. The water will",  # spill/pour
    # Buoyancy
    "float_sink":    "A heavy metal block is placed in water. It will",           # sink
    # Temperature state change
    "melt":          "An ice cube is left in a warm room. After an hour it will", # melt
    # Inertia
    "inertia":       "A book slides across a rough table. Over time it will slow", # down/stop
}

for name, p in PHYSICS_PROMPTS.items():
    print(f"[{name}] {p!r}")

[gravity_fall] 'If you let go of a ball in mid-air, it will move'
[support_remove] 'A cup rests on a table. If the table is removed, the cup will'
[collision] 'A fast ball hits a still ball head-on. The still ball starts to'
[tip_over] 'A full glass of water is tipped sideways. The water will'
[float_sink] 'A heavy metal block is placed in water. It will'
[melt] 'An ice cube is left in a warm room. After an hour it will'
[inertia] 'A book slides across a rough table. Over time it will slow'


In [ ]:
# First, a cheap check: does the model actually predict the physically-correct
# next token? Only prompts it gets 'right' have a meaningful circuit to trace.
def predicted_next(prompt):
    toks = model.tokenizer(prompt, return_tensors="pt").to('cuda')
    with torch.no_grad():
        logits = model(toks["input_ids"])
    return model.tokenizer.decode([logits[0, -1].argmax().item()])

print(f"{'prompt':16s}  -> predicted next token")
print("-" * 45)
for name, p in PHYSICS_PROMPTS.items():
    print(f"{name:16s}  -> {predicted_next(p)!r}")

prompt            -> predicted next token
---------------------------------------------
gravity_fall      -> ' in'
support_remove    -> ' fall'
collision         -> ' move'
tip_over          -> ' not'
float_sink        -> ' sink'
melt              -> ' melt'
inertia           -> ' down'


In [ ]:
# ⚠️ VERIFY: build an attribution graph per prompt. Saves each graph so a Colab
# disconnect doesn't lose work — mount Drive below if you want persistence.
import os
os.makedirs("graphs", exist_ok=True)

physics_graphs = {}
for name, prompt in PHYSICS_PROMPTS.items():
    print(f"Tracing [{name}] ...", flush=True)
    try:
        physics_graphs[name] = build_graph(prompt)
        print(f"  ok")
    except torch.cuda.OutOfMemoryError:
        print(f"  OOM on [{name}] — lower batch_size or use a smaller transcoder set.")
        torch.cuda.empty_cache()
print(f"\nCompleted {len(physics_graphs)}/{len(PHYSICS_PROMPTS)} graphs.")

Tracing [gravity_fall] ...
  ok
Tracing [support_remove] ...
  ok
Tracing [collision] ...
  ok
Tracing [tip_over] ...
  ok
Tracing [float_sink] ...
  ok
Tracing [melt] ...
  ok
Tracing [inertia] ...
  ok

Completed 7/7 graphs.


## 8  Visualise & annotate
⚠️ VERIFY: the visualisation call mirrors the tutorial. circuit-tracer can launch a local server with an interactive frontend, or you can upload graphs to Neuronpedia. On Colab the served UI needs a tunnel; the simplest route is often to save the graph file and open it on Neuronpedia.

In [ ]:
# Cross-check the exact visualisation entrypoint in
# demos/circuit_tracing_tutorial.ipynb (function name / server args differ by
# version). Pattern is typically: build graph -> create_graph_files / serve.
try:
    from circuit_tracer.frontend import serve  # ⚠️ VERIFY module path
    # serve(sanity_graph)  # uncomment once the entrypoint is confirmed
    print("Frontend import OK — confirm the serve/save entrypoint in the tutorial.")
except Exception as e:
    print("Visualisation entrypoint differs in this version — check the tutorial.")
    print("Import note:", e)

Visualisation entrypoint differs in this version — check the tutorial.
Import note: cannot import name 'serve' from 'circuit_tracer.frontend' (/usr/local/lib/python3.12/dist-packages/circuit_tracer/frontend/__init__.py)


## 9  Intervention (causal validation)
This is the step that connects to Track A: once a graph suggests a feature is doing physical reasoning, **set that feature to a different value and confirm the output changes as predicted** — the same read/write logic as the Mode-C interventions in Track A.

⚠️ VERIFY the intervention API against `demos/intervention_demo.ipynb`.

In [ ]:
# Placeholder pattern only — fill in real feature IDs after inspecting a graph.
# demos/intervention_demo.ipynb shows the exact call (typically feeding a list of
# (layer, position, feature_idx, new_value) tuples to a feature-intervention fn).
#
# Example shape (DO NOT run until feature IDs are chosen from a real graph):
# interventions = [(layer, pos, feature_idx, 0.0)]  # ablate a candidate physics feature
# new_logits = model.feature_intervention(input_ids, interventions)  # ⚠️ VERIFY name
#
# Compare argmax(new_logits) vs the clean prediction to measure causal effect,
# exactly as recovery-fraction did in Track A.
print("Intervention scaffold ready — populate feature IDs from a traced graph first.")

Intervention scaffold ready — populate feature IDs from a traced graph first.


## 10  (Optional) Persist results to Google Drive
Colab wipes local files on disconnect. Mount Drive to keep graphs across sessions.

In [ ]:
import circuit_tracer.frontend as fe
print([x for x in dir(fe) if not x.startswith('_')])

import circuit_tracer as ct
print([x for x in dir(ct) if not x.startswith('_')])

['graph_models', 'utils']
['TYPE_CHECKING', 'attribution', 'frontend', 'graph', 'replacement_model', 'transcoder', 'utils']


In [ ]:
 from google.colab import drive
 drive.mount('/content/drive')
 import shutil, os
 os.makedirs('/content/drive/MyDrive/track_b_graphs', exist_ok=True)
 for f in os.listdir('graphs'):
     shutil.copy(f'graphs/{f}', f'/content/drive/MyDrive/track_b_graphs/{f}')
 print('Saved to Drive.')

Mounted at /content/drive
Saved to Drive.


---
### Next steps after this notebook runs clean
1. Confirm the sanity circuit (Dallas→Texas→Austin) looks sensible — validates the whole pipeline.
2. For each physics prompt the model answers correctly, trace and annotate the graph.
3. Run interventions on candidate physical-reasoning features to test causal necessity.
4. Compare against Track A: is physical-reasoning structure here distributed-but-causal, as it was in the custom world models?

Record findings in `results/` and reconcile `tasks/todo.md` + `README.md`, same discipline as Track A.